In [2]:
!pip install h3 

  Using cached h3-4.5.0-cp313-cp313-win_amd64.whl.metadata (17 kB)
Using cached h3-4.5.0-cp313-cp313-win_amd64.whl (803 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Processed")
RAW_DATA = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture")
LAST_DIR = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Final")

# Point -> 50m Resolution Hexagon

In [ ]:
import geopandas as gpd
import h3
import pandas as pd
from shapely.geometry import Polygon

# 1. Load your spatial data (replace with your actual file path)
# e.g., df = pd.read_csv("your_data.csv") or gpd.read_file("your_data.geojson")
# Assuming 'df' is the DataFrame matching the structure in image_7e209c.png:
# Let's ensure it's a GeoDataFrame or has accessible geometry
gdf = gpd.read_file(DATA_DIR/"super_class_data.geojson")

# Define target H3 resolution (Resolution 10 is the closest to ~50m-75m size)
RESOLUTION = 10


def point_to_h3_hexagon(geom, res=RESOLUTION):
    if geom is None or geom.is_empty:
        return None

    # H3 functions look for (latitude, longitude) order
    lng, lat = geom.x, geom.y

    try:
        # Supports both h3-py v4 (latlng_to_cell) and v3 (geo_to_h3)
        if hasattr(h3, "latlng_to_cell"):
            h3_index = h3.latlng_to_cell(lat, lng, res)
            boundary = h3.cell_to_boundary(h3_index)
        else:
            h3_index = h3.geo_to_h3(lat, lng, res)
            boundary = h3.h3_to_geo_boundary(h3_index)

        # H3 boundary returns (lat, lng) tuples -> convert to (lng, lat) for Shapely
        flipped_boundary = [(lng, lat) for lat, lng in boundary]
        return Polygon(flipped_boundary)
    except Exception:
        return None


# 2. Convert points to hexagonal Polygons
gdf["geometry"] = gdf["geometry"].apply(point_to_h3_hexagon)

# 3. Drop any rows where conversion failed due to empty/corrupt geometry
gdf = gdf.dropna(subset=["geometry"])

# 4. Save the hex-binned dataset to a new file
# You can save as GeoJSON, Shapefile, or GeoPackage depending on your preference
output_path = DATA_DIR/"data_hexagons_50m.geojson"
gdf.to_file(output_path, driver="GeoJSON")

print(f"Successfully saved hex-binned data to {output_path}")

Successfully saved hex-binned data to C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\data_hexagons_50m.geojson


# Buffer
## Counts of shop

In [ ]:
import pandas as pd
import geopandas as gpd
import h3
import numpy as np

# 1. Load your PREVIOUSLY CONVERTED hexagon dataset
# (Replace with "data_hexagons_50m.geojson" or your exact file name)
print("Loading existing hexagon dataset...")
gdf = gpd.read_file(DATA_DIR/"data_hexagons_50m.geojson")

# Ensure CRS is correct for centroid extraction if h3_id isn't present
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

RESOLUTION = 10

# Map version-agnostic H3 functions
latlng_to_cell = getattr(h3, 'latlng_to_cell', getattr(h3, 'geo_to_h3', None))
grid_disk = getattr(h3, 'grid_disk', getattr(h3, 'k_ring', None))

# 2. Check if 'h3_id' exists. If not, generate it from the hexagon polygon centroids
if 'h3_id' not in gdf.columns:
    print("Extracting H3 string IDs from hexagon geometries...")
    gdf['h3_id'] = gdf['geometry'].apply(lambda geom: latlng_to_cell(geom.centroid.y, geom.centroid.x, RESOLUTION) if geom else None)

gdf = gdf.dropna(subset=['h3_id'])

# Define your classification targets and all context classes
target_classes = ['Restaurant', 'Hotel_Lodging', 'Cafe_Bakery', 'Malls_Department', 'Fashion_Clothing', 'Convenience_Specialty']
all_18_classes = gdf['super_class'].unique().tolist()

# 3. Create a pivot-table of cell counts per class to use as an instant lookup matrix
print("Building POI density lookup matrix from existing cells...")
poi_counts_per_cell = gdf.groupby(['h3_id', 'super_class']).size().unstack(fill_value=0)

# Ensure all 18 classes exist as columns in our lookup matrix
for cls in all_18_classes:
    if cls not in poi_counts_per_cell.columns:
        poi_counts_per_cell[cls] = 0

# 4. Isolate your target rows (These become the rows of your training matrix)
# We drop rows with duplicate (h3_id, super_class) pairs so that a single hexagon hosting 
# two different target business types duplicates cleanly into two distinct rows.
targets_df = gdf[gdf['super_class'].isin(target_classes)].copy()
targets_df = targets_df.drop_duplicates(subset=['h3_id', 'super_class'])

# Define the topological buffers
buffers = {
    '500m': 4,
    '1km': 7,
    '2km': 14
}

# 5. Compute the concentric buffer features efficiently using H3 topological rings
print("Computing multi-scale spatial buffer features via H3 lookup indices...")
feature_storage = {f"{cls}_{buf}": [] for cls in all_18_classes for buf in buffers.keys()}

# Loop through every target business location hexagon row
for idx, row in targets_df.iterrows():
    current_hex = row['h3_id']
    current_class = row['super_class']
    
    for buf_name, ring_radius in buffers.items():
        # Get all neighboring hexagon string IDs within the specified ring distance
        neighbor_cells = grid_disk(current_hex, ring_radius)
        
        # Pull the pre-calculated POI matrix for these neighboring cells and sum them up
        valid_neighbors = poi_counts_per_cell.index.intersection(neighbor_cells)
        if not valid_neighbors.empty:
            total_counts = poi_counts_per_cell.loc[valid_neighbors].sum()
        else:
            total_counts = pd.Series(0, index=all_18_classes)
            
        for cls in all_18_classes:
            count_value = total_counts[cls]
            
            # --- PROTECT AGAINST DATA LEAKAGE TRAP ---
            # If we are looking at the exact same class inside the buffer zone as our current row's target class,
            # we subtract 1 to account for the business itself.
            if cls == current_class:
                count_value = max(0, count_value - 1)
                
            feature_storage[f"{cls}_{buf_name}"].append(count_value)

# Convert our feature dictionary into a dataframe and merge it with our targets
features_df = pd.DataFrame(feature_storage)
features_df.index = targets_df.index

# Keep the geometry column as well in case you want to map things right away
final_dataset = pd.concat([targets_df[['id', 'super_class', 'h3_id', 'geometry']], features_df], axis=1)

# Downcast our integer columns to int16 to preserve your laptop's 8GB RAM
for col in features_df.columns:
    final_dataset[col] = final_dataset[col].astype('int16')

# 6. Save the final generated spatial features dataset
# Saving as GeoJSON preserves your hexagon polygons for easy downstream visual plotting!
output_path = DATA_DIR/"kathmandu_multiclass_spatial_features.geojson"
final_dataset.to_file(output_path, driver="GeoJSON")
print(f"Data Engineering complete! Saved dataset with {final_dataset.shape[1]} columns to {output_path}")

Loading existing hexagon dataset...
Extracting H3 string IDs from hexagon geometries...
Building POI density lookup matrix from existing cells...
Computing multi-scale spatial buffer features via H3 lookup indices...
Data Engineering complete! Saved dataset with 58 columns to C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_multiclass_spatial_features.geojson


## roads

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import h3
from shapely.geometry import Polygon

# ==============================================================================
# 1. Define Paths
# ==============================================================================
roads_path = DATA_DIR/"functional_roads.geojson"
hex_path = DATA_DIR/"kathmandu_multiclass_spatial_features.geojson"

print("Loading datasets...")
roads_gdf = gpd.read_file(roads_path)
hex_gdf = gpd.read_file(hex_path)

# ==============================================================================
# 2. Project to Metric Coordinate System (UTM Zone 45N)
# ==============================================================================
METRIC_CRS = "EPSG:32645" 
print(f"Projecting layers to {METRIC_CRS} for metric calculations...")
roads_utm = roads_gdf.to_crs(METRIC_CRS)
hex_utm = hex_gdf.to_crs(METRIC_CRS)

# Define target functional classes and topological buffer frameworks
road_classes = ['Arterial_Vehicular', 'Pedestrian_Only', 'Local_Commercial']
buffers = {'500m': 4, '1km': 7, '2km': 14}

# Map version-agnostic H3 function for cell disk lookup
grid_disk = getattr(h3, 'grid_disk', getattr(h3, 'k_ring', None))

# ==============================================================================
# 3. Calculate Exact Distances to each Functional Class
# ==============================================================================
print("Calculating exact distances to road classes...")
for rc in road_classes:
    print(f"-> Processing distance to: {rc}")
    # Filter lines for this specific class
    specific_roads = roads_utm[roads_utm['functional_class'] == rc]
    
    if not specific_roads.empty:
        # Combined geometry via modern .union_all() method
        combined_road_geom = specific_roads.union_all()
        # Compute shortest distance from each hexagon centroid to the road network
        hex_gdf[f'dist_to_{rc}'] = hex_utm.geometry.centroid.distance(combined_road_geom).astype('float32')
    else:
        # Correctly indented fallback if specific road class doesn't exist
        print(f"   [Warning] No features found for {rc}. Defaulting distance to 9999.")
        hex_gdf[f'dist_to_{rc}'] = 9999.0

# ==============================================================================
# 4. Calculate Multi-Scale Road Density Using H3 Topological Rings
# ==============================================================================
print("Calculating multi-scale road density using H3 networks...")

# Spatial index generation for high-speed line cutting optimization
roads_sindex = roads_utm.sindex

# Mapping from h3_id string to its physical UTM spatial boundary 
hex_lookup = dict(zip(hex_utm['h3_id'], hex_utm.geometry))

# Pre-calculate structural buffer areas to compute density values (km of road / km^2)
buffer_areas_km2 = {
    '500m': np.pi * (0.5 ** 2), # ~0.785 km^2
    '1km': np.pi * (1.0 ** 2),  # ~3.141 km^2
    '2km': np.pi * (2.0 ** 2)   # ~12.566 km^2
}

# Pre-initialize tracking dictionaries to minimize continuous frame resizing
density_results = {f'road_density_{b}': [] for b in buffers.keys()}

for idx, row in hex_gdf.iterrows():
    current_hex = row['h3_id']
    
    for b_name, ring_radius in buffers.items():
        # Identify the structural neighborhood hexagons via spatial index keys
        neighbor_ids = grid_disk(current_hex, ring_radius)
        
        # Pull actual metric polygon geometries for neighbors
        neighbor_geoms = [hex_lookup[hid] for hid in neighbor_ids if hid in hex_lookup]
        
        if neighbor_geoms:
            # Combine individual neighbor geometries into a structural envelope polygon
            combined_zone = gpd.GeoSeries(neighbor_geoms).union_all()
            
            # Fast geographic query using spatial R-tree matrix limits
            possible_matches_index = list(roads_sindex.intersection(combined_zone.bounds))
            possible_roads = roads_utm.iloc[possible_matches_index]
            
            # Perform exact geographic intersection cutting
            intersecting_roads = possible_roads.intersection(combined_zone)
            
            # Total accumulated segment lengths in kilometers
            total_length_km = intersecting_roads.length.sum() / 1000.0
            
            # Density computation
            density = total_length_km / buffer_areas_km2[b_name]
            density_results[f'road_density_{b_name}'].append(density)
        else:
            density_results[f'road_density_{b_name}'].append(0.0)

# Write output vectors back as memory-saving float32 scales
for b_name in buffers.keys():
    hex_gdf[f'road_density_{b_name}'] = np.array(density_results[f'road_density_{b_name}']).astype('float32')

# ==============================================================================
# 5. Save Output
# ==============================================================================
output_path = hex_path.replace(".geojson", "_with_roads.geojson")
hex_gdf.to_file(output_path, driver="GeoJSON")

print(f"\nProcessing Complete! All spatial road features appended successfully.")
print(f"Saved update to: {output_path}")

Loading datasets...
Projecting layers to EPSG:32645 for metric calculations...
Calculating exact distances to road classes...
-> Processing distance to: Arterial_Vehicular
-> Processing distance to: Pedestrian_Only
-> Processing distance to: Local_Commercial
Calculating multi-scale road density using H3 networks...

Processing Complete! All spatial road features appended successfully.
Saved update to: C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_multiclass_spatial_features_with_roads.geojson


# Population

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import rowcol
import h3

# 1. Define Paths
raster_path = DATA_DIR/"kathmandu_population_density.tif"
hex_path = DATA_DIR/"kathmandu_multiclass_spatial_features_with_roads.geojson"

print("Loading dataset and raster...")
hex_gdf = gpd.read_file(hex_path)

# Ensure our GeoDataFrame is in EPSG:4326 to align with H3 coordinate systems
if hex_gdf.crs != "EPSG:4326":
    hex_gdf = hex_gdf.to_crs("EPSG:4326")

# 2. Open WorldPop Raster using Rasterio
with rasterio.open(raster_path) as src:
    # Verify Raster CRS matches our Hexagons to avoid completely empty lookups
    raster_crs = src.crs.to_string() if src.crs else "Unknown"
    print(f"-> Detected Raster CRS: {raster_crs}")
    if "4326" not in raster_crs and src.crs:
        print("[CRITICAL WARNING]: Your raster is not in EPSG:4326. Index matching will fail!")
        
    pop_data = src.read(1).astype('float32')  # Read band 1
    affine_transform = src.transform
    nodata_val = src.nodata

    # Clean NaNs and NoData thresholds cleanly
    if nodata_val is not None:
        pop_data[pop_data == nodata_val] = 0.0
    pop_data[np.isnan(pop_data)] = 0.0
    pop_data[pop_data < 0] = 0.0

# 3. Define the Pixel Buffer Framework
buffers_px = {
    '500m': 5,
    '1km': 10,
    '2km': 20
}

pop_results = {f'pop_{b}': [] for b in buffers_px.keys()}

print("Pre-calculating raster pixel transformations...")
# Extract center coordinates using GeoPandas geometries (highly optimized vector approach)
centroids = hex_gdf.geometry.centroid
lngs = centroids.x.values
lats = centroids.y.values

# Compute all pixel coordinates simultaneously
rows_idx, cols_idx = rowcol(src.transform, lngs, lats)

print("Extracting multi-scale population counts...")
raster_height, raster_width = pop_data.shape

# Reset index to ensure loop index aligns with numpy array positions safely
hex_gdf = hex_gdf.reset_index(drop=True)

for idx, row in hex_gdf.iterrows():
    # Pull pre-calculated row/column pixel locations
    center_row = rows_idx[idx]
    center_col = cols_idx[idx]
    
    for b_name, px_radius in buffers_px.items():
        # Define bounding window limits for the pixel grid cut
        row_min = max(0, center_row - px_radius)
        row_max = min(raster_height, center_row + px_radius + 1)
        col_min = max(0, center_col - px_radius)
        col_max = min(raster_width, center_col + px_radius + 1)
        
        # Crop the neighborhood matrix block and sum up total population
        pixel_window = pop_data[row_min:row_max, col_min:col_max]
        total_population = float(np.sum(pixel_window))
        
        pop_results[f'pop_{b_name}'].append(total_population)

# 4. Append features to DataFrame as memory-saving float32 arrays
for b_name in buffers_px.keys():
    hex_gdf[f'pop_{b_name}'] = np.array(pop_results[f'pop_{b_name}']).astype('float32')

# 5. Save the final completed dataset
output_path = LAST_DIR/"MultiClass_DATA.geojson"
hex_gdf.to_file(output_path, driver="GeoJSON")

print("\n🎉 Architecture Pipeline Complete!")
print(f"Your final feature matrix contains {hex_gdf.shape[1]} columns.")
print(f"Saved complete master training file to: {output_path}")

Loading dataset and raster...
-> Detected Raster CRS: EPSG:4326
Pre-calculating raster pixel transformations...


C:\Users\anils\AppData\Local\Temp\ipykernel_10156\4067161692.py:48: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = hex_gdf.geometry.centroid


Extracting multi-scale population counts...

🎉 Architecture Pipeline Complete!
Your final feature matrix contains 67 columns.
Saved complete master training file to: C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_final_training_dataset.geojson


In [15]:
hex_gdf.groupby("super_class").agg({"super_class": "count"}).rename(columns={"super_class": "count"}).sort_values(by="count", ascending=False)

,count
super_class,
Restaurant,1660
Hotel_Lodging,1384
Cafe_Bakery,1031
Fashion_Clothing,964
Convenience_Specialty,771
Malls_Department,295


# multi label

In [ ]:
import os
import geopandas as gpd
import pandas as pd

# ==============================================================================
# 1. Configuration & Paths
# ==============================================================================
INPUT_PATH = LAST_DIR/"MultiClass_DATA.geojson"
OUTPUT_PATH = LAST_DIR/"MultiLabel_DATA.geojson"

print("🚀 Loading multi-class spatial dataset...")
gdf = gpd.read_file(INPUT_PATH)
print(f"Original dataset shape (Multi-Class): {gdf.shape}")

# ==============================================================================
# 2. One-Hot Encoding the Target Labels
# ==============================================================================
print("\n🔄 Generating binary indicator columns for each business class...")

# Get unique classes present in your dataset
unique_classes = gdf['super_class'].unique()
print(f"Detected classes to target: {list(unique_classes)}")

# Create binary columns prefixed with 'is_'
for cls in unique_classes:
    column_name = f"is_{cls}"
    gdf[column_name] = (gdf['super_class'] == cls).astype(int)

# ==============================================================================
# 3. Defining the Aggregation Strategy
# ==============================================================================
# Separate features to treat them correctly during the groupby step
label_cols = [f"is_{cls}" for cls in unique_classes]
metadata_to_drop = ['id', 'super_class']  # Instance IDs and the old string column are obsolete

# Build an explicit aggregation dictionary
# - Label columns: Use 'max' (1 if at least one establishment exists in that hex, else 0)
# - Spatial features & geometry: Use 'first' (since they are identical across the same h3_id)
agg_strategy = {}

for col in gdf.columns:
    if col == 'h3_id' or col in metadata_to_drop:
        continue
    elif col in label_cols:
        agg_strategy[col] = 'max'
    else:
        # This covers all spatial buffers, distances, population metrics, and the geometry column
        agg_strategy[col] = 'first'

# ==============================================================================
# 4. Compacting the Dataset by H3 Hexagon
# ==============================================================================
print("\n🗜️ Collapsing rows by unique h3_id into a multi-label framework...")
# Group by hexagon ID and apply our strategic mapping
df_flattened = gdf.groupby('h3_id', as_index=False).agg(agg_strategy)

# Re-cast back into a proper GeoDataFrame using the aggregated geometry column
gdf_multilabel = gpd.GeoDataFrame(df_flattened, geometry='geometry', crs=gdf.crs)

print(f"New multi-label dataset shape: {gdf_multilabel.shape}")

# ==============================================================================
# 5. Analyst Diagnostics Verification
# ==============================================================================
print("\n📊 --- MULTI-LABEL DISTRIBUTION CHECKS ---")
print(f"Total Unique Hexagons (Rows): {len(gdf_multilabel)}")

print("\nOccurrences per label across all hexagons:")
for col in label_cols:
    positive_count = gdf_multilabel[col].sum()
    percentage = (positive_count / len(gdf_multilabel)) * 100
    print(f" - {col}: {positive_count} hexagons ({percentage:.2f}%)")

# Calculate coexistence metrics (how many hexagons serve multiple functions?)
hex_label_counts = gdf_multilabel[label_cols].sum(axis=1)
print("\nHexagon Density Breakdown (How many labels overlap in a single cell?):")
print(hex_label_counts.value_counts().sort_index().to_frame(name='Hexagon Count'))

# ==============================================================================
# 6. Save the Output Feature Collection
# ==============================================================================
print(f"\n💾 Saving transformed multi-label GeoJSON dataset to: {OUTPUT_PATH}")
gdf_multilabel.to_file(OUTPUT_PATH, driver="GeoJSON")
print("🎉 Transformation pipeline completed successfully!")

🚀 Loading multi-class spatial dataset...
Original dataset shape (Multi-Class): (6105, 67)

🔄 Generating binary indicator columns for each business class...
Detected classes to target: ['Restaurant', 'Hotel_Lodging', 'Malls_Department', 'Cafe_Bakery', 'Fashion_Clothing', 'Convenience_Specialty']

🗜️ Collapsing rows by unique h3_id into a multi-label framework...
New multi-label dataset shape: (3435, 71)

📊 --- MULTI-LABEL DISTRIBUTION CHECKS ---
Total Unique Hexagons (Rows): 3435

Occurrences per label across all hexagons:
 - is_Restaurant: 1660 hexagons (48.33%)
 - is_Hotel_Lodging: 1384 hexagons (40.29%)
 - is_Malls_Department: 295 hexagons (8.59%)
 - is_Cafe_Bakery: 1031 hexagons (30.01%)
 - is_Fashion_Clothing: 964 hexagons (28.06%)
 - is_Convenience_Specialty: 771 hexagons (22.45%)

Hexagon Density Breakdown (How many labels overlap in a single cell?):
   Hexagon Count
1           1986
2            737
3            363
4            224
5             90
6             35

💾 Saving tr

# hexagon for the boundary

In [ ]:
import h3
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, box

# 1. Define configuration matching your pipeline parameters
RESOLUTION = 10
BBOX = [85.20, 27.60, 85.50, 27.75]  # minx, miny, maxx, maxy
OUTPUT_PATH = DATA_DIR/"kathmandu_base_grid_res10.geojson"

print("🗺️ Generating master spatial grid for bounding box...")

# 2. Build the bounding box geometry using Shapely
bbox_polygon = box(BBOX[0], BBOX[1], BBOX[2], BBOX[3])

# 3. Extract the boundary coordinates in (lat, lng) format for H3's polyfill engine
# Exterior coordinates are stored as (lng, lat) -> slice and flip them to (lat, lng)
coords = list(bbox_polygon.exterior.coords)
h3_bbox_poly = [(lat, lng) for lng, lat in coords]

# 4. Fill the polygon area with H3 hexagon indices (handles version 3 and version 4 differences)
if hasattr(h3, "polygon_to_cells"):
    # h3-py v4 standard approach
    # We wrap coordinates in a simple GeoJSON-like dict or pass it directly
    poly_shape = {"type": "Polygon", "coordinates": [[ [lng, lat] for lat, lng in h3_bbox_poly ]]}
    hex_indices = list(h3.geo_to_cells(poly_shape, RESOLUTION))
else:
    # h3-py v3 standard fallback
    poly_dict = {"type": "Polygon", "coordinates": [[ [lng, lat] for lat, lng in h3_bbox_poly ]]}
    hex_indices = list(h3.polyfill(poly_dict, RESOLUTION, geo_json_conformant=True))

print(f"📦 Generated {len(hex_indices)} base hexagons at Resolution {RESOLUTION}.")

# 5. Build geometry list from the unique indices
grid_records = []
for h3_id in hex_indices:
    # Fetch boundaries based on version
    if hasattr(h3, "cell_to_boundary"):
        boundary = h3.cell_to_boundary(h3_id)
    else:
        boundary = h3.h3_to_geo_boundary(h3_id)
        
    # Standardize boundary formatting to (lng, lat) for Shapely Polygon structure
    flipped_boundary = [(lng, lat) for lat, lng in boundary]
    hex_geom = Polygon(flipped_boundary)
    
    # Calculate centroid matches to ensure exact string alignment with your existing code
    # gdf['h3_id'] = gdf['geometry'].apply(lambda geom: latlng_to_cell(geom.centroid.y, geom.centroid.x, RESOLUTION)...)
    centroid_y, centroid_x = hex_geom.centroid.y, hex_geom.centroid.x
    
    if hasattr(h3, "latlng_to_cell"):
        verified_h3_id = h3.latlng_to_cell(centroid_y, centroid_x, RESOLUTION)
    else:
        verified_h3_id = h3.geo_to_h3(centroid_y, centroid_x, RESOLUTION)
        
    grid_records.append({
        "h3_id": verified_h3_id,
        "geometry": hex_geom
    })

# 6. Assemble into a GeoDataFrame
grid_gdf = gpd.GeoDataFrame(grid_records, crs="EPSG:4326")

# 7. Save output grid to file
grid_gdf.to_file(OUTPUT_PATH, driver="GeoJSON")
print(f"🚀 Successfully generated and saved master base grid to: {OUTPUT_PATH}")

🗺️ Generating master spatial grid for bounding box...
📦 Generated 33010 base hexagons at Resolution 10.
🚀 Successfully generated and saved master base grid to: C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_base_grid_res10.geojson


# New Columns for grid

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import h3
import numpy as np

# ==============================================================================
# 1. Load Datasets & Establish File Paths
# ==============================================================================
grid_path = DATA_DIR/"kathmandu_base_grid_res10.geojson"
computed_features_path = DATA_DIR/"kathmandu_multiclass_spatial_features.geojson"
raw_hex_path = DATA_DIR/"data_hexagons_50m.geojson"

# Saving ONLY the background empty/uncalculated grid spaces
output_path = DATA_DIR/"kathmandu_pure_background_grid.geojson"

print("🗺️ Loading master base grid and existing features...")
base_grid_gdf = gpd.read_file(grid_path)
computed_features_gdf = gpd.read_file(computed_features_path)
raw_hex_gdf = gpd.read_file(raw_hex_path)

RESOLUTION = 10
latlng_to_cell = getattr(h3, 'latlng_to_cell', getattr(h3, 'geo_to_h3', None))
grid_disk = getattr(h3, 'grid_disk', getattr(h3, 'k_ring', None))

# Generate missing h3_ids to ensure alignment across files
for df in [base_grid_gdf, computed_features_gdf, raw_hex_gdf]:
    if 'h3_id' not in df.columns:
        df['h3_id'] = df['geometry'].apply(lambda geom: latlng_to_cell(geom.centroid.y, geom.centroid.x, RESOLUTION) if geom else None)

# ==============================================================================
# 2. Isolate ONLY the Hexagons Not In Your Existing Dataset
# ==============================================================================
existing_feature_hexes = set(computed_features_gdf['h3_id'].unique())

# Drop any hexagon that exists in your training dataset to isolate pure background
empty_grid_cells = base_grid_gdf[~base_grid_gdf['h3_id'].isin(existing_feature_hexes)].copy()
print(f"🔍 Total Background Hexagons to compute (excluding existing shops): {len(empty_grid_cells)}")

# ==============================================================================
# 3. Rebuild POI Density Lookup Matrix
# ==============================================================================
all_18_classes = raw_hex_gdf['super_class'].unique().tolist()
poi_counts_per_cell = raw_hex_gdf.groupby(['h3_id', 'super_class']).size().unstack(fill_value=0)

for cls in all_18_classes:
    if cls not in poi_counts_per_cell.columns:
        poi_counts_per_cell[cls] = 0

buffers = {'500m': 4, '1km': 7, '2km': 14}
feature_cols = [f"{cls}_{buf}" for cls in all_18_classes for buf in buffers.keys()]

# ==============================================================================
# 4. Compute Buffer Features for Background Hexagons
# ==============================================================================
if len(empty_grid_cells) > 0:
    print("⚡ Computing multi-scale spatial buffers for background grid...")
    feature_storage = {col: [] for col in feature_cols}
    
    for idx, row in empty_grid_cells.iterrows():
        current_hex = row['h3_id']
        
        for buf_name, ring_radius in buffers.items():
            neighbor_cells = grid_disk(current_hex, ring_radius)
            valid_neighbors = poi_counts_per_cell.index.intersection(neighbor_cells)
            
            if not valid_neighbors.empty:
                total_counts = poi_counts_per_cell.loc[valid_neighbors].sum()
            else:
                total_counts = pd.Series(0, index=all_18_classes)
                
            for cls in all_18_classes:
                feature_storage[f"{cls}_{buf_name}"].append(total_counts[cls])
                
    empty_features_df = pd.DataFrame(feature_storage)
    empty_features_df.index = empty_grid_cells.index
    
    # Construct final matrix block
    final_background_grid = pd.concat([
        empty_grid_cells[['h3_id', 'geometry']], 
        empty_features_df
    ], axis=1)
    
    # Add minimal alignment tags
    final_background_grid['id'] = "background_cell"
    final_background_grid['super_class'] = "None"
    
    # Downcast features to keep memory footprints small for your laptop
    for col in feature_cols:
        final_background_grid[col] = final_background_grid[col].astype('int16')
        
    # Re-order columns nicely
    final_columns = ['id', 'super_class', 'h3_id', 'geometry'] + feature_cols
    final_background_grid = final_background_grid[final_columns]

    # Save to file
    final_background_grid.to_file(output_path, driver="GeoJSON")
    print(f"🎉 Complete background grid matrix successfully saved to {output_path}!")
    print(f"📊 Final Dimensions: {final_background_grid.shape[0]} Hexagon Rows | {final_background_grid.shape[1]} Feature Columns.")
else:
    print("⚠️ No new background cells found outside your existing feature dataset.")

🗺️ Loading master base grid and existing features...
🔍 Total Background Hexagons to compute (excluding existing shops): 29578
⚡ Computing multi-scale spatial buffers for background grid...
🎉 Complete background grid matrix successfully saved to C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_pure_background_grid.geojson!
📊 Final Dimensions: 29578 Hexagon Rows | 58 Feature Columns.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import h3
from shapely.geometry import Polygon

# ==============================================================================
# 1. Define Paths
# ==============================================================================
roads_path = DATA_DIR/"functional_roads.geojson"
# Input is the background grid slice from the feature generation step
hex_path = DATA_DIR/"kathmandu_pure_background_grid.geojson"
# Loaded to act as the master global geometric bounding reference dictionary
master_grid_path = DATA_DIR/"kathmandu_base_grid_res10.geojson"

print("Loading datasets...")
roads_gdf = gpd.read_file(roads_path)
hex_gdf = gpd.read_file(hex_path)
master_grid_gdf = gpd.read_file(master_grid_path)

# ==============================================================================
# 2. Project to Metric Coordinate System (UTM Zone 45N)
# ==============================================================================
METRIC_CRS = "EPSG:32645" 
print(f"Projecting layers to {METRIC_CRS} for metric calculations...")
roads_utm = roads_gdf.to_crs(METRIC_CRS)
hex_utm = hex_gdf.to_crs(METRIC_CRS)
master_grid_utm = master_grid_gdf.to_crs(METRIC_CRS)

# Ensure h3_id is populated in the master dictionary reference layer
RESOLUTION = 10
latlng_to_cell = getattr(h3, 'latlng_to_cell', getattr(h3, 'geo_to_h3', None))
if 'h3_id' not in master_grid_utm.columns:
    master_grid_utm['h3_id'] = master_grid_utm['geometry'].apply(
        lambda geom: latlng_to_cell(geom.centroid.y, geom.centroid.x, RESOLUTION) if geom else None
    )

road_classes = ['Arterial_Vehicular', 'Pedestrian_Only', 'Local_Commercial']
buffers = {'500m': 4, '1km': 7, '2km': 14}
grid_disk = getattr(h3, 'grid_disk', getattr(h3, 'k_ring', None))

# ==============================================================================
# 3. Calculate Exact Distances to each Functional Class (OPTIMIZED)
# ==============================================================================
print("Calculating exact distances to road classes using fast spatial indices...")

# Isolate the centroids of our hexagons ahead of time as a clean GeoDataFrame
hex_centroids_gdf = gpd.GeoDataFrame(
    geometry=hex_utm.geometry.centroid, 
    crs=hex_utm.crs
)
# Keep track of original index positions to map data back flawlessly
hex_centroids_gdf['original_index'] = hex_centroids_gdf.index

for rc in road_classes:
    print(f"-> Processing distance to: {rc}")
    specific_roads = roads_utm[roads_utm['functional_class'] == rc]
    
    if not specific_roads.empty:
        # High-speed nearest-neighbor search instead of computing raw massive union vectors
        nearest_join = gpd.sjoin_nearest(
            hex_centroids_gdf, 
            specific_roads[['geometry']], 
            how="left", 
            distance_col="distance_val"
        )
        
        # Resolve any rare 1-to-many match splits by taking the minimum distance per original row
        distance_series = nearest_join.groupby('original_index')['distance_val'].min()
        
        # Map values back as memory-saving float32 scales
        hex_gdf[f'dist_to_{rc}'] = distance_series.astype('float32')
    else:
        print(f"   [Warning] No features found for {rc}. Defaulting distance to 9999.")
        hex_gdf[f'dist_to_{rc}'] = 9999.0

# ==============================================================================
# 4. Calculate Multi-Scale Road Density Using H3 Topological Rings
# ==============================================================================
print("Calculating multi-scale road density using H3 networks...")

roads_sindex = roads_utm.sindex

# Build geometric dictionary from master reference grid to protect neighborhood looks
hex_lookup = dict(zip(master_grid_utm['h3_id'], master_grid_utm.geometry))

buffer_areas_km2 = {
    '500m': np.pi * (0.5 ** 2), 
    '1km': np.pi * (1.0 ** 2),  
    '2km': np.pi * (2.0 ** 2)   
}

density_results = {f'road_density_{b}': [] for b in buffers.keys()}

for idx, row in hex_gdf.iterrows():
    current_hex = row['h3_id']
    
    for b_name, ring_radius in buffers.items():
        neighbor_ids = grid_disk(current_hex, ring_radius)
        
        # Pull actual metric polygon geometries for neighbors via the master mapping dict
        neighbor_geoms = [hex_lookup[hid] for hid in neighbor_ids if hid in hex_lookup]
        
        if neighbor_geoms:
            combined_zone = gpd.GeoSeries(neighbor_geoms).union_all()
            possible_matches_index = list(roads_sindex.intersection(combined_zone.bounds))
            possible_roads = roads_utm.iloc[possible_matches_index]
            
            intersecting_roads = possible_roads.intersection(combined_zone)
            total_length_km = intersecting_roads.length.sum() / 1000.0
            
            density = total_length_km / buffer_areas_km2[b_name]
            density_results[f'road_density_{b_name}'].append(density)
        else:
            density_results[f'road_density_{b_name}'].append(0.0)

for b_name in buffers.keys():
    hex_gdf[f'road_density_{b_name}'] = np.array(density_results[f'road_density_{b_name}']).astype('float32')

# ==============================================================================
# 5. Save Output
# ==============================================================================
output_path = hex_path.replace(".geojson", "_with_roads.geojson")
hex_gdf.to_file(output_path, driver="GeoJSON")

print(f"\nProcessing Complete! All spatial road features appended successfully.")
print(f"Saved update to: {output_path}")

Loading datasets...
Projecting layers to EPSG:32645 for metric calculations...
Calculating exact distances to road classes using fast spatial indices...
-> Processing distance to: Arterial_Vehicular
-> Processing distance to: Pedestrian_Only
-> Processing distance to: Local_Commercial
Calculating multi-scale road density using H3 networks...

Processing Complete! All spatial road features appended successfully.
Saved update to: C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_pure_background_grid_with_roads.geojson


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import rowcol
import h3

# ==============================================================================
# 1. Define Paths (Updated to target your background road slice)
# ==============================================================================
raster_path = DATA_DIR/"kathmandu_population_density.tif"
# The input is now your pure background grid containing the appended road metrics
hex_path = DATA_DIR/"kathmandu_pure_background_grid_with_roads.geojson"

# Saving your final completed background dataset
output_path = DATA_DIR/"KTM_BOUNDARY_HEX.geojson"

print("Loading dataset and raster...")
hex_gdf = gpd.read_file(hex_path)

# Ensure our GeoDataFrame is in EPSG:4326 to align with H3 coordinate systems
if hex_gdf.crs != "EPSG:4326":
    hex_gdf = hex_gdf.to_crs("EPSG:4326")

# ==============================================================================
# 2. Open WorldPop Raster using Rasterio
# ==============================================================================
with rasterio.open(raster_path) as src:
    # Verify Raster CRS matches our Hexagons to avoid completely empty lookups
    raster_crs = src.crs.to_string() if src.crs else "Unknown"
    print(f"-> Detected Raster CRS: {raster_crs}")
    if "4326" not in raster_crs and src.crs:
        print("[CRITICAL WARNING]: Your raster is not in EPSG:4326. Index matching will fail!")
        
    pop_data = src.read(1).astype('float32')  # Read band 1
    affine_transform = src.transform
    nodata_val = src.nodata

    # Clean NaNs and NoData thresholds cleanly
    if nodata_val is not None:
        pop_data[pop_data == nodata_val] = 0.0
    pop_data[np.isnan(pop_data)] = 0.0
    pop_data[pop_data < 0] = 0.0

# ==============================================================================
# 3. Define the Pixel Buffer Framework
# ==============================================================================
buffers_px = {
    '500m': 5,
    '1km': 10,
    '2km': 20
}

pop_results = {f'pop_{b}': [] for b in buffers_px.keys()}

print("Pre-calculating raster pixel transformations...")
# Extract center coordinates using GeoPandas geometries (highly optimized vector approach)
centroids = hex_gdf.geometry.centroid
lngs = centroids.x.values
lats = centroids.y.values

# Compute all pixel coordinates simultaneously
rows_idx, cols_idx = rowcol(src.transform, lngs, lats)

print("Extracting multi-scale population counts...")
raster_height, raster_width = pop_data.shape

# Reset index to ensure loop index aligns with numpy array positions safely
hex_gdf = hex_gdf.reset_index(drop=True)

for idx, row in hex_gdf.iterrows():
    # Pull pre-calculated row/column pixel locations
    center_row = rows_idx[idx]
    center_col = cols_idx[idx]
    
    for b_name, px_radius in buffers_px.items():
        # Define bounding window limits for the pixel grid cut
        row_min = max(0, center_row - px_radius)
        row_max = min(raster_height, center_row + px_radius + 1)
        col_min = max(0, center_col - px_radius)
        col_max = min(raster_width, center_col + px_radius + 1)
        
        # Crop the neighborhood matrix block and sum up total population
        pixel_window = pop_data[row_min:row_max, col_min:col_max]
        total_population = float(np.sum(pixel_window))
        
        pop_results[f'pop_{b_name}'].append(total_population)

# ==============================================================================
# 4. Append features to DataFrame as memory-saving float32 arrays
# ==============================================================================
for b_name in buffers_px.keys():
    hex_gdf[f'pop_{b_name}'] = np.array(pop_results[f'pop_{b_name}']).astype('float32')

# ==============================================================================
# 5. Save the final completed dataset
# ==============================================================================
hex_gdf.to_file(output_path, driver="GeoJSON")

print("\n🎉 Background Spatial Pipeline Complete!")
print(f"Your background prediction matrix contains {hex_gdf.shape[1]} columns.")
print(f"Saved complete master background file to: {output_path}")

Loading dataset and raster...
-> Detected Raster CRS: EPSG:4326
Pre-calculating raster pixel transformations...


C:\Users\anils\AppData\Local\Temp\ipykernel_6608\2879323124.py:58: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = hex_gdf.geometry.centroid


Extracting multi-scale population counts...

🎉 Background Spatial Pipeline Complete!
Your background prediction matrix contains 67 columns.
Saved complete master background file to: C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_final_background_prediction_dataset.geojson


In [ ]:
# Drop unneeded Columns
import geopandas as gpd
df_h = gpd.read_file(LAST_DIR/"KTM_BOUNDARY_HEX.geojson")
df_h.drop(columns=(["id", "super_class"]), inplace = True)

df_t = gpd.read_file(LAST_DIR/"MultiLabel_DATA.geojson")
target_cols = [col for col in df_t.columns if col.startswith('is_')]
df_t.drop(columns=target_cols, inplace =True)

In [19]:
import pandas as pd
df_concat = pd.concat([df_h, df_t], ignore_index=True)

In [ ]:
df_concat.to_file(LAST_DIR/"hexagon_ktm.geojson", driver="geojson")

# Building Footprints

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Inputs
HEX_DATA_PATH = LAST_DIR / "MultiLabel_DATA.geojson"
BLDG_DATA_PATH = RAW_DATA / "kathmandu_raw_buildings.geojson"

# Output
OUTPUT_PATH = DATA_DIR / "kathmandu_multilabel_dataset_with_buildings.geojson"

# ==============================================================================
# 2. Load Local Datasets
# ==============================================================================
print("⏳ Loading hexagon dataset...")
gdf_hex = gpd.read_file(HEX_DATA_PATH)

print("⏳ Loading local buildings dataset (this might take a moment depending on file size)...")
bldgs_gdf = gpd.read_file(BLDG_DATA_PATH)

print(f"✅ Loaded {len(bldgs_gdf)} buildings.")

# ==============================================================================
# 3. Geometry Processing & Projection
# ==============================================================================
print("📐 Reprojecting to UTM Zone 45N (EPSG:32645) for accurate meter measurements...")
bldgs_gdf_proj = bldgs_gdf.to_crs(epsg=32645)
gdf_hex_proj = gdf_hex.to_crs(epsg=32645)

print("📏 Calculating building areas...")
bldgs_gdf_proj['bldg_area_sqm'] = bldgs_gdf_proj.geometry.area

# ==============================================================================
# 4. Spatial Join & Feature Aggregation
# ==============================================================================
print("🔗 Aggregating building footprints into hexagons...")
# Map buildings to the hexagon they intersect
joined = gpd.sjoin(bldgs_gdf_proj, gdf_hex_proj[['h3_id', 'geometry']], how='inner', predicate='intersects')

# Calculate statistics per hexagon
bldg_stats = joined.groupby('h3_id').agg(
    bldg_count=('bldg_area_sqm', 'count'),
    bldg_area_mean=('bldg_area_sqm', 'mean'),
    bldg_area_max=('bldg_area_sqm', 'max'),
    bldg_area_total=('bldg_area_sqm', 'sum')
).reset_index()

# Calculate coverage ratio based on the area of a single hexagon
hex_area = gdf_hex_proj.geometry.area.iloc[0] 
bldg_stats['bldg_coverage_ratio'] = bldg_stats['bldg_area_total'] / hex_area

# ==============================================================================
# 5. Merge & Export
# ==============================================================================
print("💾 Merging features and saving output...")
# Merge the stats back onto the original unprojected hexagon dataframe
gdf_hex = gdf_hex.merge(bldg_stats, on='h3_id', how='left')

# Fill NaN values with 0 for hexagons that contain no buildings (empty fields)
fill_cols = ['bldg_count', 'bldg_area_mean', 'bldg_area_max', 'bldg_area_total', 'bldg_coverage_ratio']
gdf_hex[fill_cols] = gdf_hex[fill_cols].fillna(0)

# Save the updated GeoJSON
gdf_hex.to_file(OUTPUT_PATH, driver="GeoJSON")

print(f"🎉 Success! Updated dataset saved to:\n{OUTPUT_PATH}")

⏳ Loading hexagon dataset...
⏳ Loading local buildings dataset (this might take a moment depending on file size)...
✅ Loaded 675877 buildings.
📐 Reprojecting to UTM Zone 45N (EPSG:32645) for accurate meter measurements...
📏 Calculating building areas...
🔗 Aggregating building footprints into hexagons...
💾 Merging features and saving output...
🎉 Success! Updated dataset saved to:
C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_multilabel_dataset_with_buildings.geojson


# DEM

In [3]:
!pip install rasterstats


   ---------------------------------------- 0/2 [simplejson]
   ---------------------------------------- 0/2 [simplejson]
   ---------------------------------------- 0/2 [simplejson]
   ---------------------------------------- 0/2 [simplejson]
   -------------------- ------------------- 1/2 [rasterstats]
   ---------------------------------------- 2/2 [rasterstats]




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
import geopandas as gpd
from pathlib import Path


# Inputs from previous steps
INPUT_GEOJSON = DATA_DIR / "kathmandu_multilabel_dataset_with_buildings.geojson"
INPUT_DEM = DATA_DIR / "kathmandu_elevation_30m.tif"  # Saved from the previous step

# Final Output
OUTPUT_GEOJSON = LAST_DIR / "MultiLabel_DATA_V2.geojson"

# Temporary warped files
WARPED_DEM = Path("kathmandu_elevation_32645.tif")
SLOPE_TIFF = Path("kathmandu_slope_32645.tif")

# ==============================================================================
# 2. Reproject DEM to Meters (EPSG:32645) for Accurate Slope
# ==============================================================================
print("🌐 Reprojecting DEM to Nepal UTM Zone 45N (EPSG:32645)...")
dst_crs = 'EPSG:32645'

with rasterio.open(INPUT_DEM) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open(WARPED_DEM, 'w', **kwargs) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

# ==============================================================================
# 3. Calculate Slope Surface via NumPy Gradient
# ==============================================================================
print("📐 Calculating slope angles in degrees...")
with rasterio.open(WARPED_DEM) as src:
    dem_arr = src.read(1)
    meta = src.meta.copy()
    
    # Get spatial resolution in meters (usually ~30m)
    res_x = abs(src.res[0])
    res_y = abs(src.res[1])
    
    # Compute horizontal and vertical gradients
    px, py = np.gradient(dem_arr, res_x, res_y)
    
    # Calculate slope magnitude
    slope_grad = np.sqrt(px**2 + py**2)
    
    # Convert rise/run gradient to degrees
    slope_deg = np.arctan(slope_grad) * (180.0 / np.pi)

# Save the calculated slope raster
with rasterio.open(SLOPE_TIFF, 'w', **meta) as dst:
    dst.write(slope_deg.astype(rasterio.float32), 1)

# ==============================================================================
# 4. Extract Zonal Statistics into Hexagons
# ==============================================================================
print("⏳ Loading your dataset with building footprints...")
gdf_hex = gpd.read_file(INPUT_GEOJSON)

# Match the hexagon projection to the meter-based rasters
gdf_hex_proj = gdf_hex.to_crs(epsg=32645)

print("📊 Computing mean elevation per hexagon...")
elev_stats = zonal_stats(
    gdf_hex_proj, 
    str(WARPED_DEM), 
    stats=['mean', 'std']
)
gdf_hex['elevation_mean'] = [stat['mean'] if stat['mean'] is not None else 0 for stat in elev_stats]
gdf_hex['elevation_std'] = [stat['std'] if stat['std'] is not None else 0 for stat in elev_stats]

print("📊 Computing mean slope per hexagon...")
slope_stats = zonal_stats(
    gdf_hex_proj, 
    str(SLOPE_TIFF), 
    stats=['mean']
)
gdf_hex['slope_mean_deg'] = [stat['mean'] if stat['mean'] is not None else 0 for stat in slope_stats]

# ==============================================================================
# 5. Save Final Export & Clean Up
# ==============================================================================
print("💾 Saving final multi-label dataset with all spatial extensions...")
gdf_hex.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

# Clean up local temporary files
if WARPED_DEM.exists(): WARPED_DEM.unlink()
if SLOPE_TIFF.exists(): SLOPE_TIFF.unlink()

print(f"\n🎉 Complete success! Final ready-to-train dataset saved to:\n{OUTPUT_GEOJSON}")

🌐 Reprojecting DEM to Nepal UTM Zone 45N (EPSG:32645)...
📐 Calculating slope angles in degrees...
⏳ Loading your dataset with building footprints...
📊 Computing mean elevation per hexagon...


c:\Users\anils\Desktop\7th SEM\env\Lib\site-packages\rasterstats\io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


📊 Computing mean slope per hexagon...
💾 Saving final multi-label dataset with all spatial extensions...

🎉 Complete success! Final ready-to-train dataset saved to:
C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\kathmandu_multilabel_dataset_final.geojson
